# Data Audit

**Project question:** What must I check before treating raw retail files as modeling data?

By the end of this notebook, you should be able to:

- separate missing, invalid, duplicated, outlying, and unmatched records
- parse numeric and date fields without hiding conversion failures
- write an auditable issue table before changing the raw data

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
sales = pd.read_csv(DATA / 'retail_sales_messy.csv')
stores = pd.read_csv(DATA / 'store_metadata.csv')
print(f'Raw rows: {len(sales)}; columns: {sales.shape[1]}')
sales.head()

The raw file stays unchanged. We create parsed copies so failed conversions become visible as missing values rather than silently changing the source.

In [ ]:
parsed = sales.copy()
for col in ['units', 'revenue', 'price', 'week', 'promotion']:
    parsed[col] = pd.to_numeric(parsed[col], errors='coerce')
parsed['date'] = pd.to_datetime(parsed['date'], errors='coerce')

audit = pd.DataFrame({
    'raw_dtype': sales.dtypes.astype(str),
    'parsed_dtype': parsed.dtypes.astype(str),
    'missing_after_parse': parsed.isna().sum(),
    'unique_nonmissing': parsed.nunique(dropna=True),
})
audit

Domain rules must be stated, not guessed. For this synthetic retailer, prices should be in dollars between 0 and 50 and weekly units should not exceed 1,000. These cutoffs would require subject-matter confirmation in a real project.

In [ ]:
known_stores = set(stores['store_id'])
issues = pd.DataFrame(index=sales.index)
issues['duplicate_record'] = sales.duplicated(keep=False)
issues['missing_required_value'] = parsed[['units', 'revenue', 'price']].isna().any(axis=1)
issues['invalid_date'] = parsed['date'].isna()
issues['nonpositive_price'] = parsed['price'].le(0)
issues['suspected_cents_price'] = parsed['price'].gt(50)
issues['extreme_units'] = parsed['units'].gt(1000)
issues['unmatched_store_key'] = ~sales['store_id'].isin(known_stores)

issue_summary = issues.sum().rename('flagged_rows').to_frame()
issue_summary

In [ ]:
flagged_rows = pd.concat([sales, issues], axis=1).loc[issues.any(axis=1)]
flagged_rows

**Interpretation:** Duplicate flags identify both copies, so the flagged-row total is not the number of rows to remove. A negative price is invalid; a price above $50 is a suspected unit error; an extreme units value is a review item, not automatic proof of an error.

**Transfer exercise:** Write three domain rules for your project data. For each, state whether a violation should be corrected, excluded, or retained with a warning, and identify who can authorize that decision.